In [48]:
import os
import pandas as pd

# === CONFIG ===
trade_folder = r"C:\Users\naray\OneDrive\Desktop\assignment\2024_04_01_T"

quote_folder = r"C:\Users\naray\OneDrive\Desktop\assignment\2024_04_01_Q"

# === COLUMNS from documentation ===
trade_columns = [
    'date', 'time', 'price', 'volume', 'aggressor_side', 'trade_period',
    'trade_id', 'buyer', 'buy_algo_type', 'buy_order_capacity',
    'seller', 'sell_algo_type', 'sell_order_capacity'
]

quote_columns = [
    'date', 'time', 'bid_price', 'bid_size', 'ask_price', 'ask_size'
]

def extract_company(filename, suffix):
    return filename.replace(f"_2024_04_01_{suffix}.asc", "")


In [49]:
# === Load and Combine Trade Files ===
trade_dfs = []
for file in os.listdir(trade_folder):
    if file.endswith('_T.asc'):
        company = extract_company(file, "T")
        df = pd.read_csv(os.path.join(trade_folder, file), header=None, names=trade_columns)
        df['company'] = company
        trade_dfs.append(df)

combined_trade = pd.concat(trade_dfs)
combined_trade.to_csv("combined_trades.csv", index=False)
print("✅ Saved combined_trades.csv")


✅ Saved combined_trades.csv


In [50]:
# === Load and Combine Quote Files ===
quote_dfs = []
for file in os.listdir(quote_folder):
    if file.endswith('_Q.asc'):
        company = extract_company(file, "Q")
        df = pd.read_csv(os.path.join(quote_folder, file), header=None, names=quote_columns)
        df['company'] = company
        quote_dfs.append(df)

combined_quote = pd.concat(quote_dfs)
combined_quote.to_csv("combined_quotes.csv", index=False)
print("✅ Saved combined_quotes.csv")


✅ Saved combined_quotes.csv


In [51]:
# === Load combined files ===
trades = pd.read_csv("combined_trades.csv")
quotes = pd.read_csv("combined_quotes.csv")

# === Convert timestamp and round to minute ===
trades['timestamp'] = pd.to_datetime(trades['date'] + ' ' + trades['time'])
trades['minute'] = trades['timestamp'].dt.floor('min')

quotes['timestamp'] = pd.to_datetime(quotes['date'] + ' ' + quotes['time'])
quotes['minute'] = quotes['timestamp'].dt.floor('min')


In [52]:
# === Trade aggregation: OHLC, total_volume, weighted price ===
agg_trades = trades.groupby(['company', 'minute']).agg(
    o=('price', 'first'),
    h=('price', 'max'),
    l=('price', 'min'),
    c=('price', 'last'),
    total_volume=('volume', 'sum'),
    num_trades=('price', 'count'),
    weighted_price=('price', lambda x: (x * trades.loc[x.index, 'volume']).sum() / trades.loc[x.index, 'volume'].sum()),
).reset_index()


In [53]:
# === Quote aggregation: spread stats, bid/ask size ===
quotes['spread'] = quotes['ask_price'] - quotes['bid_price']

agg_quotes = quotes.groupby(['company', 'minute']).agg(
    avg_spread=('spread', 'mean'),
    max_spread=('spread', 'max'),
    min_spread=('spread', 'min'),
    total_bid_size=('bid_size', 'sum'),
    total_ask_size=('ask_size', 'sum'),
    weighted_avg_bid_price=('bid_price', lambda x: (x * quotes.loc[x.index, 'bid_size']).sum() / quotes.loc[x.index, 'bid_size'].sum() if quotes.loc[x.index, 'bid_size'].sum() != 0 else 0),
    weighted_avg_ask_price=('ask_price', lambda x: (x * quotes.loc[x.index, 'ask_size']).sum() / quotes.loc[x.index, 'ask_size'].sum() if quotes.loc[x.index, 'ask_size'].sum() != 0 else 0),
).reset_index()


In [54]:
# 1. Merge trade and quote data
merged = pd.merge(agg_trades, agg_quotes, on=['company', 'minute'], how='inner')
merged = merged.sort_values(['company', 'minute']).reset_index(drop=True)

# 2. Add rolling features (for example, 5 and 10-minute windows)
for window in [5, 10]:
    merged[f'rolling_mean_{window}'] = merged.groupby('company')['c'].transform(
        lambda x: x.rolling(window, min_periods=1).mean()
    )
    merged[f'rolling_std_{window}'] = merged.groupby('company')['c'].transform(
        lambda x: x.rolling(window, min_periods=1).std().fillna(0)
    )

# 3. Create the target: whether price went up next minute
merged['future_price'] = merged.groupby('company')['c'].shift(-1)
merged['target'] = (merged['future_price'] > merged['c']).astype(int)

# 4. Drop rows with missing values (e.g., due to shift or std)
merged.dropna(inplace=True)

# 5. Split into train and test sets (last 100 minutes per company as test)
train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)


C:\Users\naray\AppData\Local\Temp\ipykernel_25284\814757046.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
C:\Users\naray\AppData\Local\Temp\ipykernel_25284\814757046.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)


In [55]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import numpy as np

# === Step 1: Feature + Target Already Done ===
# merged has rolling features, target, and no leakage

feature_cols = [col for col in merged.columns if col.startswith('rolling') or col in [
    'total_volume', 'num_trades', 'weighted_price', 'avg_spread', 'max_spread', 'min_spread',
    'o', 'h', 'l', 'c',  # OHLC prices
]]

X_train = train[feature_cols]
y_train = train['target']
X_test = test[feature_cols]
y_test = test['target']

# === Step 3: Train Model ===
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# === Step 4: Predict Probabilities ===
probs = model.predict_proba(X_test)[:, 1]

# === Step 5: Apply Threshold (e.g., 0.6 for “only sure buys”) ===
threshold = 0.6
preds = (probs > threshold).astype(int)

# === Step 6: Check Precision (Important as per your request) ===
precision = precision_score(y_test, preds, zero_division=0)
print(f'Precision at threshold {threshold}: {precision:.4f}')


Precision at threshold 0.6: 0.5076


In [56]:
print(train['target'].value_counts())


target
0    4575
1    3962
Name: count, dtype: int64


In [57]:
# Example: get predicted probabilities and apply threshold
probs = model.predict_proba(test[feature_cols])[:,1]
preds = (probs > 0.6).astype(int)

print(pd.Series(preds).value_counts())


0    2773
1     327
Name: count, dtype: int64


In [58]:


# preds and test['target'] already defined

# Confusion matrix: returns [ [TN, FP], [FN, TP] ]
tn, fp, fn, tp = confusion_matrix(test['target'], preds).ravel()

print(f"True Positives (TP): {tp}")
print(f"False Positives (FP): {fp}")
print(f"True Negatives (TN): {tn}")
print(f"False Negatives (FN): {fn}")

precision = precision_score(test['target'], preds)
recall = recall_score(test['target'], preds)
f1 = f1_score(test['target'], preds)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


True Positives (TP): 166
False Positives (FP): 161
True Negatives (TN): 1487
False Negatives (FN): 1286
Precision: 0.5076
Recall: 0.1143
F1 Score: 0.1866


In [59]:


initial_capital = 3000
trade_amount = 200
capital = initial_capital
positions = []

# preds = model predictions with your threshold (e.g. 0.6)
# test = your test dataframe with 'c' (close price) and 'future_price' columns
for i in range(len(preds)):
    if preds[i] == 1:
        current_price = test.iloc[i]['c']
        future_price = test.iloc[i]['future_price']
        if not np.isnan(future_price):
            return_pct = (future_price - current_price) / current_price
            profit = return_pct * trade_amount
            capital += profit
            positions.append(profit)
        else:
            positions.append(0)
    else:
        positions.append(0)

print(f"Final capital after backtest: {capital:.2f}")
print(f"Total trades: {sum(preds)}")

# Metrics
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

y_true = test['target']
precision = precision_score(y_true, preds, zero_division=0)
recall = recall_score(y_true, preds, zero_division=0)
f1 = f1_score(y_true, preds, zero_division=0)
cm = confusion_matrix(y_true, preds)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Confusion Matrix:\n{cm}")

# Optional: loop over thresholds to see precision & trade counts
for threshold in [0.5, 0.6, 0.7]:
    preds = (probs > threshold).astype(int)
    precision = precision_score(y_true, preds, zero_division=0)
    total_trades = preds.sum()
    print(f"Threshold: {threshold}, Precision: {precision:.4f}, Trades: {total_trades}")


Final capital after backtest: 3002.38
Total trades: 327
Precision: 0.5076
Recall: 0.1143
F1 Score: 0.1866
Confusion Matrix:
[[1487  161]
 [1286  166]]
Threshold: 0.5, Precision: 0.4795, Trades: 1149
Threshold: 0.6, Precision: 0.5076, Trades: 327
Threshold: 0.7, Precision: 0.3962, Trades: 53


In [60]:
for threshold in [0.5, 0.6, 0.65, 0.7]:
    preds = (probs > threshold).astype(int)
    precision = precision_score(y_test, preds, zero_division=0)
    total_trades = preds.sum()
    print(f"Threshold: {threshold}, Precision: {precision:.4f}, Trades: {total_trades}")


Threshold: 0.5, Precision: 0.4795, Trades: 1149
Threshold: 0.6, Precision: 0.5076, Trades: 327
Threshold: 0.65, Precision: 0.4500, Trades: 140
Threshold: 0.7, Precision: 0.3962, Trades: 53


In [61]:
rolling_windows = [3, 5, 10]
cols_to_roll = ['o', 'h', 'l', 'c', 'total_volume', 'avg_spread']

for window in rolling_windows:
    for col in cols_to_roll:
        merged[f'{col}_rolling_mean_{window}'] = merged.groupby('company')[col].transform(
            lambda x: x.rolling(window, min_periods=1).mean())
        merged[f'{col}_rolling_std_{window}'] = merged.groupby('company')[col].transform(
            lambda x: x.rolling(window, min_periods=1).std().fillna(0))


In [62]:
horizons = [1, 3, 5]

for horizon in horizons:
    # Future price after 'horizon' minutes
    merged[f'future_price_{horizon}'] = merged.groupby('company')['c'].shift(-horizon)
    # Price change ratio
    merged[f'future_return_{horizon}'] = (merged[f'future_price_{horizon}'] - merged['c']) / merged['c']


In [63]:
# 1. Define rolling windows and horizon steps
rolling_windows = [3, 5, 10]
horizons = [1, 3, 5]

# 2. Columns to apply rolling stats on
cols_to_roll = ['o', 'h', 'l', 'c', 'total_volume', 'avg_spread']

# 3. Add rolling mean and std features grouped by company
for window in rolling_windows:
    for col in cols_to_roll:
        merged[f'{col}_rolling_mean_{window}'] = merged.groupby('company')[col].transform(
            lambda x: x.rolling(window, min_periods=1).mean())
        merged[f'{col}_rolling_std_{window}'] = merged.groupby('company')[col].transform(
            lambda x: x.rolling(window, min_periods=1).std().fillna(0))

# 4. Add horizon future prices and returns (shifted to avoid leakage)
for horizon in horizons:
    merged[f'future_price_{horizon}'] = merged.groupby('company')['c'].shift(-horizon)
    merged[f'future_return_{horizon}'] = (merged[f'future_price_{horizon}'] - merged['c']) / merged['c']

# 5. Define target based on next-minute price movement (horizon=1)
merged['target'] = (merged['future_return_1'] > 0).astype(int)

# 6. Drop rows with NaNs created due to shifting
merged.dropna(subset=[f'future_price_{h}' for h in horizons], inplace=True)

# 7. Define your feature columns
feature_cols = [col for col in merged.columns if (
    col.startswith('rolling') or 
    col in ['o', 'h', 'l', 'c', 'total_volume', 'avg_spread', 'max_spread', 'min_spread',
            'total_bid_size', 'total_ask_size', 'weighted_avg_bid_price', 'weighted_avg_ask_price']
)]

# 8. Split into train and test (last 100 per company for test)
train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)


C:\Users\naray\AppData\Local\Temp\ipykernel_25284\3127491955.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
C:\Users\naray\AppData\Local\Temp\ipykernel_25284\3127491955.py:36: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)


In [71]:


# Assuming 'merged' is your dataframe, already merged and sorted

# Step 1: Add multiple rolling windows on 'c' (close price) and 'total_volume' if exists
rolling_windows = [3, 5, 10, 20]

for w in rolling_windows:
    merged[f'rolling_mean_c_{w}'] = merged.groupby('company')['c'].transform(lambda x: x.rolling(w, min_periods=1).mean())
    merged[f'rolling_std_c_{w}'] = merged.groupby('company')['c'].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    
    if 'total_volume' in merged.columns:
        merged[f'rolling_mean_volume_{w}'] = merged.groupby('company')['total_volume'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        merged[f'rolling_std_volume_{w}'] = merged.groupby('company')['total_volume'].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

# Step 2: Prepare feature columns list
feature_cols = [col for col in merged.columns if (
    col.startswith('rolling_mean') or 
    col.startswith('rolling_std') or 
    col in ['o', 'h', 'l', 'c', 'avg_spread', 'max_spread', 'min_spread', 
            'total_bid_size', 'total_ask_size', 'weighted_avg_bid_price', 'weighted_avg_ask_price']
)]

# Step 3: Split train/test as before
train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)

# Step 4: Train Random Forest
X_train = train[feature_cols]
y_train = train['target']

X_test = test[feature_cols]
y_test = test['target']

model = RandomForestClassifier(n_estimators=80,  random_state=42)
model.fit(X_train, y_train)

# Step 5: Predict with probability threshold 
probs = model.predict_proba(X_test)[:,1]
threshold = 0.63
preds = (probs > threshold).astype(int)

# Step 6: Evaluate Precision and Confusion Matrix
precision = precision_score(y_test, preds, zero_division=0)
cm = confusion_matrix(y_test, preds)

print(f"Precision at threshold {threshold}: {precision:.4f}")
print("Confusion Matrix:")
print(cm)
print("Predicted 1 count:", preds.sum())
print("Actual 1 count:", y_test.sum())


C:\Users\naray\AppData\Local\Temp\ipykernel_25284\3113205914.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
C:\Users\naray\AppData\Local\Temp\ipykernel_25284\3113205914.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)


Precision at threshold 0.63: 0.6044
Confusion Matrix:
[[1613   36]
 [1396   55]]
Predicted 1 count: 91
Actual 1 count: 1451


In [72]:


# Assuming 'merged' is your dataframe, already merged and sorted

# Step 1: Add multiple rolling windows on 'c' (close price) and 'total_volume' if exists
rolling_windows = [3, 5, 10, 20]

for w in rolling_windows:
    merged[f'rolling_mean_c_{w}'] = merged.groupby('company')['c'].transform(lambda x: x.rolling(w, min_periods=1).mean())
    merged[f'rolling_std_c_{w}'] = merged.groupby('company')['c'].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))
    
    if 'total_volume' in merged.columns:
        merged[f'rolling_mean_volume_{w}'] = merged.groupby('company')['total_volume'].transform(lambda x: x.rolling(w, min_periods=1).mean())
        merged[f'rolling_std_volume_{w}'] = merged.groupby('company')['total_volume'].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

# Step 2: Prepare feature columns list
feature_cols = [col for col in merged.columns if (
    col.startswith('rolling_mean') or 
    col.startswith('rolling_std') or 
    col in ['o', 'h', 'l', 'c', 'avg_spread', 'max_spread', 'min_spread', 
            'total_bid_size', 'total_ask_size', 'weighted_avg_bid_price', 'weighted_avg_ask_price']
)]

# Step 3: Split train/test as before
train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)

# Step 4: Train Random Forest
X_train = train[feature_cols]
y_train = train['target']

X_test = test[feature_cols]
y_test = test['target']

model = RandomForestClassifier(n_estimators=100,  random_state=42)
model.fit(X_train, y_train)

# Step 5: Predict with probability threshold 
probs = model.predict_proba(X_test)[:,1]
threshold = 0.63
preds = (probs > threshold).astype(int)

# Step 6: Evaluate Precision and Confusion Matrix
precision = precision_score(y_test, preds, zero_division=0)
cm = confusion_matrix(y_test, preds)

print(f"Precision at threshold {threshold}: {precision:.4f}")
print("Confusion Matrix:")
print(cm)
print("Predicted 1 count:", preds.sum())
print("Actual 1 count:", y_test.sum())


C:\Users\naray\AppData\Local\Temp\ipykernel_25284\2908860640.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train = merged.groupby('company').apply(lambda x: x.iloc[:-100]).reset_index(drop=True)
C:\Users\naray\AppData\Local\Temp\ipykernel_25284\2908860640.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  test = merged.groupby('company').apply(lambda x: x.iloc[-100:]).reset_index(drop=True)


Precision at threshold 0.63: 0.6143
Confusion Matrix:
[[1622   27]
 [1408   43]]
Predicted 1 count: 70
Actual 1 count: 1451
